# 💬 Restaurant Sentiment Analysis
### LA Luxury Restaurant Recommendation System — Phase 4

**Purpose:** Layer two complementary NLP analyses on top of the classified restaurant dataset:

| Layer | Columns Added | Model | Input Text |
|-------|--------------|-------|------------|
| **Emotion Scoring** | `emotion_joy` … `emotion_disgust`, `dominant_emotion` | `j-hartmann/emotion-english-distilroberta-base` | `restaurant_metadata` (sentence-level, aggregated) |
| **Sentiment Tone** | `overall_sentiment`, `sentiment_score` | `cardiffnlp/twitter-roberta-base-sentiment-latest` | `Description` (single-shot) |
| **Dining Mood** | `dining_mood` | Rule-based keyword logic | `Description` + `restaurant_metadata` |

**Why restaurant sentiment differs from book sentiment:**  
Restaurant descriptions are single sentences (avg 22 words), unlike book descriptions  
which are multi-paragraph. We therefore use `restaurant_metadata` (10–11 structured  
sentences, 417–518 chars) as the primary emotion input — split by period into  
sentence chunks — and `Description` for overall tone classification.

**Input:**  `restaurants_with_classifications.csv` + `tagged_restaurant_descriptions.txt`  
**Output:** `restaurants_with_emotions.csv` — the final fully-enriched dataset for the Gradio UI

---
**Pipeline Overview:**
```
restaurants_with_classifications.csv
        │
        ├── restaurant_metadata  ──► j-hartmann emotion model
        │   (split by '.' → sentence chunks)   │
        │                                       ▼
        │                          7 per-sentence scores
        │                          aggregated → max per emotion
        │                                       │
        │                          emotion_joy / sadness / anger
        │                          emotion_fear / neutral / surprise
        │                          emotion_disgust + dominant_emotion
        │
        ├── Description  ──────► cardiffnlp sentiment model
        │   (full single-shot)           │
        │                                ▼
        │                  overall_sentiment (Positive/Neutral/Negative)
        │                  sentiment_score   (float 0.0–1.0)
        │
        └── Description + metadata  ──► keyword logic
                                                │
                                                ▼
                                    dining_mood label
                                    (6 categories)
        │
        ▼
  restaurants_with_emotions.csv  (final enriched dataset)
```

## 📦 Cell 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm

import torch
from transformers import pipeline

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 240)

# Auto-detect best available compute device
if torch.cuda.is_available():
    DEVICE = 0
    device_label = f"GPU — {torch.cuda.get_device_name(0)}"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    device_label = "Apple Silicon MPS"
else:
    DEVICE = "cpu"
    device_label = "CPU"

print(f"✅ Libraries loaded.")
print(f"   PyTorch version  : {torch.__version__}")
print(f"   Inference device : {device_label}")
print(f"   Note: CPU is fine — 71 restaurants processes in ~3 minutes per model.")

✅ Libraries loaded.
   PyTorch version  : 2.10.0+cpu
   Inference device : CPU
   Note: CPU is fine — 71 restaurants processes in ~3 minutes per model.


## 📂 Cell 2 — Load the Classified Restaurant Dataset

> **Note:** Place `restaurants_with_classifications.csv` and  
> `tagged_restaurant_descriptions.txt` in the same folder as this notebook.

In [ ]:
CSV_PATH = "../data/restaurants_with_classifications.csv"
TXT_PATH = "../data/tagged_restaurant_descriptions.txt"

restaurants = pd.read_csv(CSV_PATH)

# Load tagged descriptions text file (one restaurant per line)
with open(TXT_PATH, "r", encoding="utf-8") as f:
    tagged_lines = [line.strip().strip('"') for line in f.readlines() if line.strip()]

print(f"✅ Dataset loaded: {len(restaurants)} restaurants, {len(restaurants.columns)} columns")
print(f"✅ Tagged descriptions loaded: {len(tagged_lines)} lines")
print(f"\nExisting columns from Phase 3:")
for i, col in enumerate(restaurants.columns, 1):
    print(f"   {i:>2}. {col}")

✅ Dataset loaded: 71 restaurants, 21 columns
✅ Tagged descriptions loaded: 71 lines

Existing columns from Phase 3:
    1. Name
    2. Location
    3. Description
    4. Address
    5. Telephone Number
    6. Price
    7. Cuisine Type
    8. Dining Atmosphere
    9. Sky-High Rooftop
   10. Michelin-Guide
   11. Customer Ratings
   12. Operation Hours
   13. Reservations
   14. Dress Code
   15. restaurant_metadata
   16. simple_cuisine_group
   17. dining_format
   18. predicted_occasion
   19. occasion_confidence
   20. predicted_vibe
   21. vibe_confidence


## 👀 Cell 3 — Preview Text Fields Used for Sentiment Analysis

Two text sources will be used:  
- `restaurant_metadata` — 10–11 structured sentences, 417–518 chars → emotion scoring  
- `Description` — 1–2 prose sentences, ~22 words avg → overall sentiment tone

In [3]:
print("=== TEXT FIELD STATS ===")
restaurants['desc_word_count'] = restaurants['Description'].str.split().str.len()
restaurants['meta_char_count'] = restaurants['restaurant_metadata'].str.len()
restaurants['meta_sent_count'] = restaurants['restaurant_metadata'].str.split('.').apply(
    lambda x: len([s for s in x if len(s.strip()) > 10])
)

print(f"Description — avg words   : {restaurants['desc_word_count'].mean():.1f} "
      f"(min {restaurants['desc_word_count'].min()}, max {restaurants['desc_word_count'].max()})")
print(f"Metadata    — avg chars   : {restaurants['meta_char_count'].mean():.0f} "
      f"(min {restaurants['meta_char_count'].min()}, max {restaurants['meta_char_count'].max()})")
print(f"Metadata    — avg sentences: {restaurants['meta_sent_count'].mean():.1f} "
      f"(consistent across all 71 restaurants)")
print(f"\n⚠️  Model token limit (distilroberta): 512 tokens ≈ ~2,000 chars")
print(f"   Longest metadata: {restaurants['meta_char_count'].max()} chars — safely within limit ✅")

print(f"\n=== SAMPLE TEXT FIELDS (3 restaurants) ===")
for idx in [0, 13, 26]:  # Somni, Holbox, Maccheroni Republic
    row = restaurants.iloc[idx]
    print(f"\n[{idx}] {row['Name']}  ({row['Michelin-Guide']}, {row['Price']})")
    print(f"  Description : {row['Description']}")
    print(f"  Metadata    : {row['restaurant_metadata'][:200]}...")

=== TEXT FIELD STATS ===
Description — avg words   : 22.2 (min 16, max 30)
Metadata    — avg chars   : 454 (min 417, max 518)
Metadata    — avg sentences: 10.1 (consistent across all 71 restaurants)

⚠️  Model token limit (distilroberta): 512 tokens ≈ ~2,000 chars
   Longest metadata: 518 chars — safely within limit ✅

=== SAMPLE TEXT FIELDS (3 restaurants) ===

[0] Somni  (3-Star, $$$$$)
  Description : A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor Zabala with one seating per night.
  Metadata    : Somni is a Spanish Modernist restaurant located in West Hollywood, Los Angeles. A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor Zabala with one seating ...

[13] Holbox  (1-Star, $$)
  Description : A Yucatecan-style seafood counter from Chef Gilberto Cetina Jr. inside Mercado La Paloma offering some of LA's most delicious and affordable Michelin-starred food.
  Metadata    : Holbox is a Mexica

---
## 🎭 LAYER 1 — Emotion Scoring (j-hartmann Model)

**Model:** `j-hartmann/emotion-english-distilroberta-base`  
**Detects:** 7 Ekman emotions — `joy`, `sadness`, `anger`, `fear`, `neutral`, `surprise`, `disgust`  
**Strategy:** Split each restaurant's `restaurant_metadata` by period into sentence chunks,  
run the model on all sentences at once, then take the **max score per emotion** across  
all sentences. This captures the emotional *peak* of the dining description.  

**Why max and not mean?**  
A single sentence like *"breathtaking panoramic views"* signals peak `joy` more  
reliably than the average across structural sentences like *"Price range: \$\$\$\$"*.

## ⚙️ Cell 4 — Load the Emotion Classification Model

In [4]:
EMOTION_MODEL = "j-hartmann/emotion-english-distilroberta-base"

print(f"⏳ Loading emotion model: {EMOTION_MODEL}")
print("   First run downloads ~330MB — subsequent runs load from HuggingFace cache.")
print()

emotion_classifier = pipeline(
    "text-classification",
    model=EMOTION_MODEL,
    top_k=None,         # Return all 7 emotion scores per input
    device=DEVICE,
    truncation=True,    # Safety: truncate if any input exceeds 512 tokens
    max_length=512
)

# Smoke test on a restaurant-flavoured sentence
test_sentence = "A breathtaking candlelit dining room with sweeping panoramic views of Los Angeles."
test_result = emotion_classifier(test_sentence)
top_emotion = sorted(test_result[0], key=lambda x: x['score'], reverse=True)[0]

print(f"✅ Emotion model loaded on device: {DEVICE}")
print(f"\nSmoke test: '{test_sentence[:60]}...'")
print(f"  Top emotion: {top_emotion['label']} ({top_emotion['score']:.4f}) ✅")
print(f"  All scores:")
for item in sorted(test_result[0], key=lambda x: x['score'], reverse=True):
    bar = '█' * int(item['score'] * 20)
    print(f"    {item['label']:<12} {bar} {item['score']:.4f}")

⏳ Loading emotion model: j-hartmann/emotion-english-distilroberta-base
   First run downloads ~330MB — subsequent runs load from HuggingFace cache.



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Emotion model loaded on device: cpu

Smoke test: 'A breathtaking candlelit dining room with sweeping panoramic...'
  Top emotion: joy (0.9142) ✅
  All scores:
    joy          ██████████████████ 0.9142
    surprise      0.0374
    fear          0.0217
    neutral       0.0159
    disgust       0.0052
    anger         0.0032
    sadness       0.0024


## 🛠️ Cell 5 — Define the Sentence Splitting & Emotion Aggregation Helpers

In [5]:
# The 7 emotion labels from j-hartmann model (alphabetical order for consistent indexing)
EMOTION_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]


def split_metadata_to_sentences(metadata: str, min_words: int = 4) -> list:
    """
    Splits a restaurant_metadata string into meaningful sentence chunks.

    Filters out fragments that are too short (structural tags like
    'Price range: $$$$' split mid-tag) to keep only informative sentences.

    Parameters
    ----------
    metadata  : The full restaurant_metadata string
    min_words : Minimum word count for a sentence to be included (default 4)

    Returns
    -------
    list : Cleaned list of sentence strings
    """
    raw_sentences = metadata.split('.')
    cleaned = []
    for sent in raw_sentences:
        sent = sent.strip()
        # Skip structural tag fragments (e.g. 'Price range: $$$$')
        if len(sent.split()) >= min_words:
            cleaned.append(sent)
    return cleaned if cleaned else [metadata]  # Fallback: use full text


def calculate_max_emotion_scores(predictions: list) -> dict:
    """
    Aggregates sentence-level emotion predictions to restaurant level
    by taking the MAX score per emotion across all sentences.

    This captures the emotional PEAK of the description rather than
    the average, which is diluted by structural/neutral sentences.

    Parameters
    ----------
    predictions : List of prediction dicts from emotion_classifier
                  Each element is a list of {label, score} dicts for one sentence

    Returns
    -------
    dict : {emotion_label: max_score} for all 7 emotions
    """
    per_emotion_scores = {label: [] for label in EMOTION_LABELS}

    for sentence_prediction in predictions:
        # Sort alphabetically so index lines up with EMOTION_LABELS
        sorted_preds = sorted(sentence_prediction, key=lambda x: x["label"])
        for idx, label in enumerate(EMOTION_LABELS):
            per_emotion_scores[label].append(sorted_preds[idx]["score"])

    return {label: float(np.max(scores)) for label, scores in per_emotion_scores.items()}


# Test the helpers
test_meta = restaurants.loc[0, 'restaurant_metadata']  # Somni
test_sentences = split_metadata_to_sentences(test_meta)
print(f"=== SENTENCE SPLIT TEST — {restaurants.loc[0, 'Name']} ===")
print(f"Input length : {len(test_meta)} chars")
print(f"Sentences    : {len(test_sentences)}")
for i, s in enumerate(test_sentences, 1):
    print(f"  [{i}] {s}")

test_preds = emotion_classifier(test_sentences)
test_scores = calculate_max_emotion_scores(test_preds)
print(f"\nAggregated max emotion scores:")
for label, score in sorted(test_scores.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(score * 20)
    print(f"  {label:<12} {bar} {score:.4f}")

print(f"\n✅ Helper functions ready.")

=== SENTENCE SPLIT TEST — Somni ===
Input length : 427 chars
Sentences    : 7
  [1] Somni is a Spanish Modernist restaurant located in West Hollywood, Los Angeles
  [2] A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor Zabala with one seating per night
  [3] Michelin Guide: Michelin 3-Star
  [4] 0/5 Cuisine Group: Spanish
  [5] Dining Format: Tasting Menu / Omakase
  [6] Best For: Special Occasion
  [7] Vibe: Refined & Elegant

Aggregated max emotion scores:
  neutral      ██████████████████ 0.9140
  joy          █████████ 0.4879
  disgust      ████████ 0.4099
  surprise     ████ 0.2162
  sadness       0.0377
  fear          0.0125
  anger         0.0120

✅ Helper functions ready.


## 🔄 Cell 6 — Dry Run: Emotion Scoring on First 5 Restaurants

Validates the pipeline on a small sample before running all 71.

In [6]:
print("=== DRY RUN: First 5 restaurants ===")
print(f"{'Restaurant':<35} {'joy':>7} {'neutral':>8} {'surprise':>9} {'sadness':>8} → dominant")
print("-" * 82)

for i in range(5):
    meta = restaurants.loc[i, 'restaurant_metadata']
    sentences = split_metadata_to_sentences(meta)
    preds = emotion_classifier(sentences)
    scores = calculate_max_emotion_scores(preds)
    dominant = max(scores, key=scores.get)
    name = restaurants.loc[i, 'Name']
    print(
        f"  {name:<33} "
        f"{scores['joy']:>7.4f} "
        f"{scores['neutral']:>8.4f} "
        f"{scores['surprise']:>9.4f} "
        f"{scores['sadness']:>8.4f} → {dominant}"
    )

print("\n✅ Dry run successful. Running full dataset in Cell 7.")

=== DRY RUN: First 5 restaurants ===
Restaurant                              joy  neutral  surprise  sadness → dominant
----------------------------------------------------------------------------------
  Somni                              0.4879   0.9140    0.2162   0.0377 → neutral
  Providence                         0.5403   0.9576    0.0756   0.0103 → neutral
  Hayato                             0.7157   0.9178    0.0662   0.0109 → neutral
  n/naka                             0.9471   0.9178    0.1084   0.0377 → joy
  Melisse                            0.4539   0.9178    0.0692   0.0377 → neutral

✅ Dry run successful. Running full dataset in Cell 7.


## 🔄 Cell 7 — Run Emotion Scoring on All 71 Restaurants

In [9]:
print("⏳ Running emotion analysis on all 71 restaurants...")
print(f"   Model : {EMOTION_MODEL}")
print(f"   Input : restaurant_metadata (sentence-split, max-aggregated)")
print()

# NOTE: We track by row_index (0..70) instead of Name to safely handle the
# two restaurants that intentionally appear twice in the dataset:
#   • Pine & Crane  (rows 34 and 35 — DTLA and Silver Lake locations)
#   • Badmaash      (rows 51 and 52 — Hollywood and DTLA locations)
# Merging on Name causes a MergeError because those names are not unique.
row_index_tracker = []
emotion_scores_collector = {label: [] for label in EMOTION_LABELS}

for i in tqdm(range(len(restaurants)), desc="Emotion Scoring"):
    metadata = restaurants.loc[i, 'restaurant_metadata']
    sentences = split_metadata_to_sentences(metadata)
    predictions = emotion_classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)

    row_index_tracker.append(i)
    for label in EMOTION_LABELS:
        emotion_scores_collector[label].append(round(max_scores[label], 4))

print(f"\n✅ Emotion scoring complete for all {len(restaurants)} restaurants.")
print(f"   Tracked by row index: {len(row_index_tracker)} entries")

⏳ Running emotion analysis on all 71 restaurants...
   Model : j-hartmann/emotion-english-distilroberta-base
   Input : restaurant_metadata (sentence-split, max-aggregated)



Emotion Scoring: 100%|██████████| 71/71 [00:06<00:00, 10.74it/s]


✅ Emotion scoring complete for all 71 restaurants.
   Tracked by row index: 71 entries


## 🔗 Cell 8 — Build the Emotion DataFrame & Merge

Creates a clean emotion scores DataFrame and merges it back into  
the main `restaurants` DataFrame on restaurant `Name`.

In [10]:
# Build emotion DataFrame
emotion_df = pd.DataFrame(emotion_scores_collector)
emotion_df['row_index'] = row_index_tracker  # guaranteed unique join key

# Cast to float32
emotion_df[EMOTION_LABELS] = emotion_df[EMOTION_LABELS].astype('float32')

# Prefix emotion columns
emotion_df = emotion_df.rename(columns={
    label: f'emotion_{label}' for label in EMOTION_LABELS
})

# Derive dominant_emotion
emotion_cols_prefixed = [f'emotion_{label}' for label in EMOTION_LABELS]
emotion_df['dominant_emotion'] = (
    emotion_df[emotion_cols_prefixed]
    .idxmax(axis=1)
    .str.replace('emotion_', '', regex=False)
)

print(f"✅ Emotion DataFrame built: {emotion_df.shape}")
print(emotion_df[['row_index', 'dominant_emotion'] + emotion_cols_prefixed].head(5).to_string(index=False))

# Merge back using row_index — safe for all 71 rows including duplicates
restaurants = (
    restaurants
    .reset_index()
    .rename(columns={'index': 'row_index'})
    .merge(emotion_df, on='row_index', how='left')
    .drop(columns=['row_index'])
    .reset_index(drop=True)
)

print(f"\n✅ Merged. Shape: {restaurants.shape}")
print("Verify Pine & Crane (expect 2 rows, each with own scores):")
print(restaurants[restaurants['Name'] == 'Pine & Crane'][['Name', 'Location', 'dominant_emotion']].to_string())
print("Verify Badmaash (expect 2 rows, each with own scores):")
print(restaurants[restaurants['Name'] == 'Badmaash'][['Name', 'Location', 'dominant_emotion']].to_string())

✅ Emotion DataFrame built: (71, 9)
 row_index dominant_emotion  emotion_anger  emotion_disgust  emotion_fear  emotion_joy  emotion_neutral  emotion_sadness  emotion_surprise
         0          neutral         0.0120           0.4099        0.0125       0.4879           0.9140           0.0377            0.2162
         1          neutral         0.0259           0.0264        0.0057       0.5403           0.9576           0.0103            0.0756
         2          neutral         0.0097           0.0369        0.0060       0.7157           0.9178           0.0109            0.0662
         3              joy         0.0120           0.4099        0.0060       0.9471           0.9178           0.0377            0.1084
         4          neutral         0.0120           0.4099        0.0060       0.4539           0.9178           0.0377            0.0692

✅ Merged. Shape: (71, 32)
Verify Pine & Crane (expect 2 rows, each with own scores):
            Name              Location domina

## 📋 Cell 9 — Emotion Scores: Full Results Table

In [11]:
print("=== EMOTION SCORES — ALL RESTAURANTS ===")
display_cols = [
    'Name', 'Michelin-Guide', 'Price',
    'emotion_joy', 'emotion_neutral', 'emotion_surprise',
    'emotion_sadness', 'emotion_anger', 'emotion_fear',
    'emotion_disgust', 'dominant_emotion'
]
print(restaurants[display_cols].to_string(index=True))

=== EMOTION SCORES — ALL RESTAURANTS ===
                              Name     Michelin-Guide  Price  emotion_joy  emotion_neutral  emotion_surprise  emotion_sadness  emotion_anger  emotion_fear  emotion_disgust dominant_emotion
0                            Somni             3-Star  $$$$$       0.4879           0.9140            0.2162           0.0377         0.0120        0.0125           0.4099          neutral
1                       Providence             3-Star   $$$$       0.5403           0.9576            0.0756           0.0103         0.0259        0.0057           0.0264          neutral
2                           Hayato             2-Star   $$$$       0.7157           0.9178            0.0662           0.0109         0.0097        0.0060           0.0369          neutral
3                           n/naka             2-Star   $$$$       0.9471           0.9178            0.1084           0.0377         0.0120        0.0060           0.4099              joy
4             

## 📊 Cell 10 — Dominant Emotion Distribution

In [12]:
print("=== DOMINANT EMOTION DISTRIBUTION ===")
dom_counts = restaurants['dominant_emotion'].value_counts()
for emotion, count in dom_counts.items():
    bar = '█' * count
    print(f"  {emotion:<12} | {bar} ({count})")

print(f"\n=== AVERAGE EMOTION SCORES (all 71 restaurants) ===")
avg_scores = restaurants[emotion_cols_prefixed].mean().sort_values(ascending=False)
for col, val in avg_scores.items():
    label = col.replace('emotion_', '')
    bar = '█' * int(val * 30)
    print(f"  {label:<12} | {bar} {val:.4f}")

print(f"\n=== DOMINANT EMOTION BY MICHELIN TIER ===")
michelin_order = ['3-Star', '2-Star', '1-Star', 'Bib-Gourmand', 'Michelin-Selected', 'No']
print(pd.crosstab(
    pd.Categorical(restaurants['Michelin-Guide'], categories=michelin_order, ordered=True),
    restaurants['dominant_emotion']
).to_string())

=== DOMINANT EMOTION DISTRIBUTION ===
  neutral      | █████████████████████████████████████████████████████████████████ (65)
  joy          | ██████ (6)

=== AVERAGE EMOTION SCORES (all 71 restaurants) ===
  neutral      | ███████████████████████████ 0.9323
  joy          | ██████████████████████ 0.7344
  disgust      | ███████ 0.2360
  surprise     | ██ 0.0951
  sadness      |  0.0275
  anger        |  0.0141
  fear         |  0.0108

=== DOMINANT EMOTION BY MICHELIN TIER ===
dominant_emotion   joy  neutral
row_0                          
3-Star               0        2
2-Star               1        3
1-Star               1       19
Bib-Gourmand         1        9
Michelin-Selected    2        9
No                   1       23


## 🧪 Cell 11 — Emotion Accuracy Check: Logical Expectations

Validates emotion predictions against known restaurant characteristics.

In [13]:
print("=== EMOTION ACCURACY SPOT-CHECK ===")
print()

# Expectation 1: Rooftop / view restaurants should have high joy or surprise
rooftop = restaurants[restaurants['Sky-High Rooftop'] == 'Yes']
joy_or_surprise = {'joy', 'surprise', 'neutral'}
print("Rooftop/view restaurants — expected high joy or surprise:")
for _, row in rooftop.iterrows():
    status = "✅" if row['dominant_emotion'] in joy_or_surprise else "⚠️ "
    print(f"  {status} {row['Name']:<32} → dominant: {row['dominant_emotion']:<12} "
          f"joy={row['emotion_joy']:.3f}  surprise={row['emotion_surprise']:.3f}")

# Expectation 2: Avant-garde/theatrical restaurants should lean toward surprise
print()
theatrical = restaurants[restaurants['predicted_vibe'] == 'Theatrical & Experiential']
print("Theatrical/Experiential restaurants — expected surprise or joy:")
for _, row in theatrical.iterrows():
    status = "✅" if row['dominant_emotion'] in {'surprise', 'joy', 'neutral'} else "⚠️ "
    print(f"  {status} {row['Name']:<32} → dominant: {row['dominant_emotion']:<12} "
          f"surprise={row['emotion_surprise']:.3f}  joy={row['emotion_joy']:.3f}")

# Expectation 3: Top 10 by joy score
print()
print("Top 10 restaurants by emotion_joy score:")
top_joy = restaurants.nlargest(10, 'emotion_joy')[['Name', 'Michelin-Guide', 'emotion_joy', 'dominant_emotion']]
for _, row in top_joy.iterrows():
    print(f"  {row['Name']:<35} ({row['Michelin-Guide']:<18}) joy={row['emotion_joy']:.4f}  dominant={row['dominant_emotion']}")

# Accuracy
checks = []
for _, row in rooftop.iterrows():
    checks.append(row['dominant_emotion'] in joy_or_surprise)
for _, row in theatrical.iterrows():
    checks.append(row['dominant_emotion'] in {'surprise', 'joy', 'neutral'})

accuracy = sum(checks) / len(checks)
print(f"\n=== SPOT-CHECK ACCURACY: {accuracy:.1%} ({sum(checks)}/{len(checks)} passed) ===")

=== EMOTION ACCURACY SPOT-CHECK ===

Rooftop/view restaurants — expected high joy or surprise:
  ✅ 71Above                          → dominant: neutral      joy=0.862  surprise=0.072
  ✅ La Boucherie                     → dominant: neutral      joy=0.924  surprise=0.094
  ✅ Aperture at City Club LA         → dominant: neutral      joy=0.743  surprise=0.204
  ✅ Elephante                        → dominant: neutral      joy=0.915  surprise=0.076
  ✅ Yamashiro                        → dominant: neutral      joy=0.759  surprise=0.149
  ✅ LouLou                           → dominant: neutral      joy=0.737  surprise=0.063

Theatrical/Experiential restaurants — expected surprise or joy:
  ✅ Kali                             → dominant: neutral      surprise=0.074  joy=0.807
  ✅ Mastro's Ocean Club              → dominant: neutral      surprise=0.070  joy=0.928
  ✅ Niku X                           → dominant: neutral      surprise=0.070  joy=0.817
  ✅ H&H Brazilian Steakhouse         → dominant:

---
## 👍 LAYER 2 — Overall Sentiment Tone (cardiffnlp Model)

**Model:** `cardiffnlp/twitter-roberta-base-sentiment-latest`  
**Detects:** `Positive` / `Neutral` / `Negative` sentiment  
**Input:** `Description` field (the crisp one-sentence prose summary)  

**Why the Description and not metadata?**  
The `Description` contains the original editorial prose — adjective-rich language  
written to evoke a reaction. The `restaurant_metadata` adds structured labels  
(`Price range: $$$$`, `Michelin Guide: 3-Star`) that dilute sentiment tone.  
Using `Description` gives a purer, more accurate sentiment read.

**Output columns:**
- `overall_sentiment` — `Positive` / `Neutral` / `Negative` label
- `sentiment_score`   — Confidence float (0.0 – 1.0) for the predicted label

## ⚙️ Cell 12 — Load the Sentiment Tone Model

In [14]:
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

print(f"⏳ Loading sentiment model: {SENTIMENT_MODEL}")
print("   First run downloads ~500MB — subsequent runs load from HuggingFace cache.")
print()

sentiment_classifier = pipeline(
    "text-classification",
    model=SENTIMENT_MODEL,
    device=DEVICE,
    truncation=True,
    max_length=512
)

# Smoke test on restaurant descriptions
smoke_tests = [
    "A breathtaking fine dining restaurant with sweeping panoramic views from Malibu to the mountains.",
    "A cozy neighbourhood bistro serving rustic handmade pastas in a warm casual setting.",
    "An avant-garde sci-fi dreamscape dining experience inside a wavy obelisk architectural marvel.",
]
print("✅ Sentiment model loaded.")
print("\nSmoke tests:")
for test in smoke_tests:
    result = sentiment_classifier(test)[0]
    label_clean = result['label'].replace('LABEL_0', 'Negative').replace('LABEL_1', 'Neutral').replace('LABEL_2', 'Positive')
    # cardiffnlp latest model uses Negative/Neutral/Positive labels directly
    print(f"  '{test[:70]}...'")
    print(f"    → {result['label']} ({result['score']:.4f})")

⏳ Loading sentiment model: cardiffnlp/twitter-roberta-base-sentiment-latest
   First run downloads ~500MB — subsequent runs load from HuggingFace cache.



config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Sentiment model loaded.

Smoke tests:
  'A breathtaking fine dining restaurant with sweeping panoramic views fr...'
    → positive (0.9513)
  'A cozy neighbourhood bistro serving rustic handmade pastas in a warm c...'
    → positive (0.8002)
  'An avant-garde sci-fi dreamscape dining experience inside a wavy obeli...'
    → positive (0.5081)


## 🛠️ Cell 13 — Define the Sentiment Normalisation Helper

The cardiffnlp model may return either named labels (`Positive`) or  
numeric labels (`LABEL_2`) depending on version. This helper normalises  
both formats to consistent `Positive` / `Neutral` / `Negative` strings.

In [15]:
# Label normalisation map for cardiffnlp model variants
SENTIMENT_LABEL_MAP = {
    # Named labels (current default)
    'Positive':  'Positive',
    'Neutral':   'Neutral',
    'Negative':  'Negative',
    # Numeric labels (some model versions)
    'LABEL_2':   'Positive',
    'LABEL_1':   'Neutral',
    'LABEL_0':   'Negative',
    # Lowercase variants
    'positive':  'Positive',
    'neutral':   'Neutral',
    'negative':  'Negative',
}


def classify_sentiment(text: str) -> tuple:
    """
    Runs the cardiffnlp sentiment classifier on a single text string.

    Parameters
    ----------
    text : Restaurant Description text (1–2 sentences)

    Returns
    -------
    tuple : (sentiment_label: str, confidence_score: float)
            sentiment_label is normalised to 'Positive'/'Neutral'/'Negative'
    """
    if not text or len(str(text).strip()) == 0:
        return ('Neutral', 0.0)

    result = sentiment_classifier(str(text))[0]
    raw_label = result['label']
    normalised_label = SENTIMENT_LABEL_MAP.get(raw_label, 'Neutral')
    return (normalised_label, round(result['score'], 4))


# Test on a spread of restaurant descriptions
test_indices = [0, 13, 26, 36, 44, 51]
print("=== SENTIMENT CLASSIFICATION TESTS ===")
print(f"{'Restaurant':<35} {'Sentiment':<12} {'Score':<8} Description")
print("-" * 110)
for idx in test_indices:
    row = restaurants.iloc[idx]
    label, score = classify_sentiment(row['Description'])
    print(f"  {row['Name']:<33} {label:<12} {score:<8.4f} {row['Description'][:60]}...")

print(f"\n✅ classify_sentiment() ready.")

=== SENTIMENT CLASSIFICATION TESTS ===
Restaurant                          Sentiment    Score    Description
--------------------------------------------------------------------------------------------------------------
  Somni                             Neutral      0.7853   A 14-seat Spanish Modernist chef's counter offering a 20+ co...
  Holbox                            Positive     0.8666   A Yucatecan-style seafood counter from Chef Gilberto Cetina ...
  Maccheroni Republic               Neutral      0.5199   A Michelin Bib Gourmand Italian-American trattoria offering ...
  71Above                           Positive     0.8417   A breathtaking contemporary American fine dining restaurant ...
  Capo                              Positive     0.8742   An elegant candlelit Italian restaurant in Santa Monica offe...
  Badmaash                          Neutral      0.7255   A Michelin Selected modern Indian restaurant blending bold s...

✅ classify_sentiment() ready.


## 🔄 Cell 14 — Run Sentiment Tone on All 71 Restaurants

In [16]:
print("⏳ Running sentiment analysis on all 71 restaurants...")
print(f"   Model : {SENTIMENT_MODEL}")
print(f"   Input : Description column (one-sentence prose)")
print()

overall_sentiments = []
sentiment_scores   = []

for i in tqdm(range(len(restaurants)), desc="Sentiment Analysis"):
    description = restaurants.loc[i, 'Description']
    label, score = classify_sentiment(description)
    overall_sentiments.append(label)
    sentiment_scores.append(score)

restaurants['overall_sentiment'] = overall_sentiments
restaurants['sentiment_score']   = sentiment_scores

print(f"\n✅ Sentiment analysis complete.")
print(f"\n=== OVERALL SENTIMENT DISTRIBUTION ===")
sent_counts = restaurants['overall_sentiment'].value_counts()
for sentiment, count in sent_counts.items():
    bar = '█' * count
    print(f"  {sentiment:<12} | {bar} ({count})")
print(f"\n  Avg confidence: {restaurants['sentiment_score'].mean():.4f}")

⏳ Running sentiment analysis on all 71 restaurants...
   Model : cardiffnlp/twitter-roberta-base-sentiment-latest
   Input : Description column (one-sentence prose)



Sentiment Analysis: 100%|██████████| 71/71 [00:02<00:00, 26.76it/s]



✅ Sentiment analysis complete.

=== OVERALL SENTIMENT DISTRIBUTION ===
  Positive     | ██████████████████████████████████████████████████ (50)
  Neutral      | █████████████████████ (21)

  Avg confidence: 0.7267


## 📋 Cell 15 — Sentiment Results: Full Table

In [17]:
print("=== SENTIMENT RESULTS — ALL RESTAURANTS ===")
sent_cols = ['Name', 'Michelin-Guide', 'Price', 'Dining Atmosphere',
             'overall_sentiment', 'sentiment_score', 'Description']
print(restaurants[sent_cols].to_string(index=True))

=== SENTIMENT RESULTS — ALL RESTAURANTS ===
                              Name     Michelin-Guide  Price        Dining Atmosphere overall_sentiment  sentiment_score                                                                                                                                                                                          Description
0                            Somni             3-Star  $$$$$              Fine-Dining           Neutral           0.7853                                                                   A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor Zabala with one seating per night.
1                       Providence             3-Star   $$$$              Fine-Dining          Positive           0.8813                                 A celebrated seafood-forward fine dining institution from Chef Michael Cimarusti with nearly 20 years of Michelin recognition and a focus on West Coast ingredients.
2 

## 🧪 Cell 16 — Sentiment Accuracy Check

In [18]:
print("=== SENTIMENT ACCURACY SPOT-CHECK ===")
print()

# Expectation: 5-star rated restaurants should be Positive
five_star = restaurants[restaurants['Customer Ratings'] == 5.0]
print(f"Restaurants rated 5.0 — expected Positive sentiment:")
checks = []
for _, row in five_star.iterrows():
    status = "✅" if row['overall_sentiment'] == 'Positive' else "⚠️ "
    checks.append(row['overall_sentiment'] == 'Positive')
    print(f"  {status} {row['Name']:<35} → {row['overall_sentiment']:<10} ({row['sentiment_score']:.4f})")

# Expectation: Negative sentiment restaurants should have lower ratings
print()
negatives = restaurants[restaurants['overall_sentiment'] == 'Negative']
if negatives.empty:
    print("ℹ️  No restaurants classified as Negative sentiment.")
else:
    print(f"Restaurants with Negative sentiment:")
    for _, row in negatives.iterrows():
        print(f"  ⚠️  {row['Name']:<35} → {row['overall_sentiment']:<10} rating={row['Customer Ratings']}")
        print(f"      Description: {row['Description']}")

# Sentiment vs Rating cross-check
print()
print("=== AVERAGE RATING BY SENTIMENT LABEL ===")
avg_rating_by_sent = restaurants.groupby('overall_sentiment')['Customer Ratings'].agg(['mean', 'count'])
avg_rating_by_sent.columns = ['avg_rating', 'count']
print(avg_rating_by_sent.to_string())
print("  (Positive > Neutral > Negative would confirm model alignment with ratings)")

accuracy = sum(checks) / len(checks)
print(f"\n=== 5-STAR SENTIMENT ACCURACY: {accuracy:.1%} ({sum(checks)}/{len(checks)} correct) ===")

=== SENTIMENT ACCURACY SPOT-CHECK ===

Restaurants rated 5.0 — expected Positive sentiment:
  ⚠️  Somni                               → Neutral    (0.7853)
  ✅ Providence                          → Positive   (0.8813)
  ⚠️  Hayato                              → Neutral    (0.6084)
  ✅ n/naka                              → Positive   (0.5908)
  ✅ Vespertine                          → Positive   (0.6789)
  ✅ Sushi Kaneyoshi                     → Positive   (0.7699)
  ✅ Restaurant Ki                       → Positive   (0.6682)
  ⚠️  715 Sushi                           → Neutral    (0.5515)
  ✅ Morihiro                            → Positive   (0.8747)
  ✅ Holbox                              → Positive   (0.8666)
  ⚠️  Sushi Inaba                         → Neutral    (0.6565)
  ⚠️  Mori Nozomi                         → Neutral    (0.6285)
  ✅ CUT by Wolfgang Puck                → Positive   (0.8974)
  ✅ Chi Spacca                          → Positive   (0.8317)
  ✅ Capo                      

---
## 🌡️ LAYER 3 — Dining Mood (Keyword-Based)

Derives a human-readable `dining_mood` label from keyword patterns across  
both `Description` and `restaurant_metadata`. This gives the Gradio UI a  
natural-language mood tag that users can relate to — independent of the  
statistical emotion scores.

**Dining Mood Labels (6 categories):**
| Label | Signal Keywords | Example Restaurants |
|-------|----------------|--------------------|
| `Warm & Inviting` | warm, cozy, welcoming, hospitality, beloved | Capo, Maccheroni Republic |
| `Dramatic & Exciting` | avant-garde, theatrical, innovative, bold, sci-fi | Vespertine, Meteora |
| `Elegant & Refined` | elegant, refined, sophisticated, luxurious, opulent | Providence, Mélisse |
| `Intimate & Personal` | intimate, hidden, exclusive, tucked, serene | Hayato, Sushi Kaneyoshi |
| `Lively & Energetic` | energetic, lively, buzzy, vibrant, live entertainment | Mastro's, Parks BBQ |
| `Relaxed & Casual` | casual, relaxed, rustic, unpretentious, affordable | Holbox, Langer's |

## 🛠️ Cell 17 — Define the Dining Mood Classification Logic

In [19]:
# Keyword sets for each dining mood — ordered by priority (top = highest priority)
DINING_MOOD_RULES = [
    (
        "Dramatic & Exciting",
        ['avant-garde', 'theatrical', 'sci-fi', 'dreamscape', 'architectural marvel',
         'obelisk', 'primordial', 'bold', 'innovative', 'foraged', 'experimental',
         'ancient history', 'theatrical', 'immersive', 'dramatic']
    ),
    (
        "Intimate & Personal",
        ['intimate', 'hidden', 'tucked', 'exclusive', 'serene', 'subterranean',
         'counter setting', 'counter seat', '7-seat', '10-seat', '14-seat',
         'hard-to-find', 'backroom', 'one seating', 'minimalist', 'personal']
    ),
    (
        "Warm & Inviting",
        ['warm', 'cozy', 'cosy', 'welcoming', 'hospitality', 'beloved', 'neighbourhood',
         'candlelit', 'rustic', 'craftsman', 'bungalow', 'homemade', 'house-made',
         'family', 'comfort', 'nostalgic', 'iconic']
    ),
    (
        "Elegant & Refined",
        ['elegant', 'refined', 'sophisticated', 'luxurious', 'luxury', 'opulent',
         'exquisite', 'acclaimed', 'prestigious', 'premium', 'world-class',
         'sleek', 'polished', 'impeccable', 'white-glove']
    ),
    (
        "Lively & Energetic",
        ['lively', 'energetic', 'buzzy', 'vibrant', 'live entertainment', 'energetic',
         'communal', 'shareable', 'loud', 'bar', 'party', 'celebration',
         'lounge', 'social', 'tableside grilling', 'grilling']
    ),
    (
        "Relaxed & Casual",
        ['casual', 'relaxed', 'affordable', 'unpretentious', 'laid-back',
         'approachable', 'fun', 'simple', 'counter', 'food hall', 'street food']
    ),
]


def classify_dining_mood(row: pd.Series) -> str:
    """
    Classifies the dining mood of a restaurant based on keyword matching.

    Searches both Description and restaurant_metadata for signal keywords.
    Rules are checked in priority order — first match wins.
    Defaults to 'Elegant & Refined' if no keywords match (appropriate for
    an upscale LA restaurant dataset where elegance is the baseline).

    Parameters
    ----------
    row : DataFrame row with 'Description' and 'restaurant_metadata' columns

    Returns
    -------
    str : One of the 6 dining mood labels
    """
    combined_text = (
        str(row['Description']).lower() + ' ' +
        str(row['restaurant_metadata']).lower()
    )

    for mood_label, keywords in DINING_MOOD_RULES:
        if any(kw in combined_text for kw in keywords):
            return mood_label

    # Default for upscale restaurant dataset
    return 'Elegant & Refined'


restaurants['dining_mood'] = restaurants.apply(classify_dining_mood, axis=1)

print("=== DINING MOOD DISTRIBUTION ===")
mood_counts = restaurants['dining_mood'].value_counts()
for mood, count in mood_counts.items():
    bar = '█' * count
    print(f"  {mood:<28} | {bar} ({count})")

print(f"\n✅ dining_mood assigned for all {len(restaurants)} restaurants.")

=== DINING MOOD DISTRIBUTION ===
  Elegant & Refined            | ███████████████████████ (23)
  Warm & Inviting              | ██████████████████ (18)
  Dramatic & Exciting          | ███████████████ (15)
  Intimate & Personal          | █████████████ (13)
  Lively & Energetic           | ██ (2)

✅ dining_mood assigned for all 71 restaurants.


## 📋 Cell 18 — Dining Mood Full Review

In [20]:
print("=== DINING MOOD — ALL RESTAURANTS ===")
print(f"{'Restaurant':<35} {'Mood':<30} {'Michelin':<18} {'Vibe'}")
print("-" * 100)
for _, row in restaurants[['Name', 'dining_mood', 'Michelin-Guide', 'predicted_vibe']].iterrows():
    print(f"  {row['Name']:<33} {row['dining_mood']:<28} {row['Michelin-Guide']:<18} {row['predicted_vibe']}")

=== DINING MOOD — ALL RESTAURANTS ===
Restaurant                          Mood                           Michelin           Vibe
----------------------------------------------------------------------------------------------------
  Somni                             Intimate & Personal          3-Star             Refined & Elegant
  Providence                        Elegant & Refined            3-Star             Hip & Trendy
  Hayato                            Intimate & Personal          2-Star             Intimate
  n/naka                            Intimate & Personal          2-Star             Refined & Elegant
  Melisse                           Intimate & Personal          2-Star             Refined & Elegant
  Vespertine                        Dramatic & Exciting          2-Star             Hip & Trendy
  Sushi Kaneyoshi                   Intimate & Personal          1-Star             Intimate
  Restaurant Ki                     Dramatic & Exciting          1-Star             

---
## 📊 COMBINED REVIEW — All Sentiment Layers

## 🗂️ Cell 19 — Full Sentiment Summary Table

In [21]:
print("=== COMPLETE SENTIMENT RESULTS — ALL 71 RESTAURANTS ===")
summary_cols = [
    'Name', 'Michelin-Guide', 'Price',
    'dominant_emotion', 'overall_sentiment', 'sentiment_score',
    'dining_mood'
]
restaurants[summary_cols]

=== COMPLETE SENTIMENT RESULTS — ALL 71 RESTAURANTS ===


,Name,Michelin-Guide,Price,dominant_emotion,overall_sentiment,sentiment_score,dining_mood
0,Somni,3-Star,$$$$$,neutral,Neutral,0.7853,Intimate & Personal
1,Providence,3-Star,$$$$,neutral,Positive,0.8813,Elegant & Refined
2,Hayato,2-Star,$$$$,neutral,Neutral,0.6084,Intimate & Personal
3,n/naka,2-Star,$$$$,joy,Positive,0.5908,Intimate & Personal
4,Melisse,2-Star,$$$$,neutral,Positive,0.6538,Intimate & Personal
...,...,...,...,...,...,...,...
66,Elephante,Michelin-Selected,$$$$,neutral,Positive,0.9044,Lively & Energetic
67,Nobu Malibu,No,$$$$$,neutral,Positive,0.8943,Warm & Inviting
68,Nobu Los Angeles,No,$$$$$,neutral,Positive,0.6237,Intimate & Personal
69,Yamashiro,No,$$$$,neutral,Positive,0.6071,Warm & Inviting


## 📈 Cell 20 — Cross-Tabulations: Sentiment × Classifications

In [22]:
print("=== DOMINANT EMOTION × DINING MOOD ===")
print(pd.crosstab(
    restaurants['dominant_emotion'],
    restaurants['dining_mood']
).to_string())

print("\n=== OVERALL SENTIMENT × MICHELIN TIER ===")
michelin_order = ['3-Star', '2-Star', '1-Star', 'Bib-Gourmand', 'Michelin-Selected', 'No']
print(pd.crosstab(
    pd.Categorical(restaurants['Michelin-Guide'], categories=michelin_order, ordered=True),
    restaurants['overall_sentiment']
).to_string())

print("\n=== DINING MOOD × CUISINE GROUP ===")
print(pd.crosstab(
    restaurants['simple_cuisine_group'],
    restaurants['dining_mood']
).to_string())

=== DOMINANT EMOTION × DINING MOOD ===
dining_mood       Dramatic & Exciting  Elegant & Refined  Intimate & Personal  Lively & Energetic  Warm & Inviting
dominant_emotion                                                                                                  
joy                                 1                  3                    1                   0                1
neutral                            14                 20                   12                   2               17

=== OVERALL SENTIMENT × MICHELIN TIER ===
overall_sentiment  Neutral  Positive
row_0                               
3-Star                   1         1
2-Star                   1         3
1-Star                  10        10
Bib-Gourmand             4         6
Michelin-Selected        3         8
No                       2        22

=== DINING MOOD × CUISINE GROUP ===
dining_mood            Dramatic & Exciting  Elegant & Refined  Intimate & Personal  Lively & Energetic  Warm & Inviting
simpl

## 🏆 Cell 21 — Top Restaurants by Emotion Category

Practical view: the highest-scoring restaurant per emotion and mood label.

In [23]:
print("=== TOP RESTAURANT BY EMOTION SCORE ===")
for emotion in EMOTION_LABELS:
    col = f'emotion_{emotion}'
    top_row = restaurants.loc[restaurants[col].idxmax()]
    print(f"  Highest {emotion:<12}: {top_row['Name']:<35} ({top_row['Michelin-Guide']}, "
          f"{top_row['Price']}) score={top_row[col]:.4f}")

print()
print("=== TOP 5 BY JOY SCORE ===")
top_joy = restaurants.nlargest(5, 'emotion_joy')[['Name', 'Michelin-Guide', 'Price',
                                                    'emotion_joy', 'dining_mood']]
print(top_joy.to_string(index=False))

print()
print("=== TOP 5 BY SURPRISE SCORE ===")
top_surprise = restaurants.nlargest(5, 'emotion_surprise')[['Name', 'Michelin-Guide', 'Price',
                                                              'emotion_surprise', 'dining_mood']]
print(top_surprise.to_string(index=False))

print()
print("=== MOST POSITIVE SENTIMENT RESTAURANTS ===")
top_pos = restaurants[restaurants['overall_sentiment'] == 'Positive'].nlargest(
    10, 'sentiment_score'
)[['Name', 'Michelin-Guide', 'overall_sentiment', 'sentiment_score', 'dining_mood']]
print(top_pos.to_string(index=False))

=== TOP RESTAURANT BY EMOTION SCORE ===
  Highest anger       : CUT by Wolfgang Puck                (No, $$$$) score=0.0791
  Highest disgust     : Somni                               (3-Star, $$$$$) score=0.4099
  Highest fear        : Restaurant Ki                       (1-Star, $$$$) score=0.0479
  Highest joy         : The Arthur J                        (No, $$$$) score=0.9845
  Highest neutral     : Providence                          (3-Star, $$$$) score=0.9576
  Highest sadness     : Somni                               (3-Star, $$$$$) score=0.0377
  Highest surprise    : Restaurant Ki                       (1-Star, $$$$) score=0.2944

=== TOP 5 BY JOY SCORE ===
         Name    Michelin-Guide Price  emotion_joy         dining_mood
 The Arthur J                No  $$$$       0.9845   Elegant & Refined
    Rasarumah      Bib-Gourmand   $$$       0.9589     Warm & Inviting
Gucci Osteria            1-Star  $$$$       0.9572   Elegant & Refined
    Majordomo Michelin-Selected   $$$ 

## 🔍 Cell 22 — Full Restaurant Profile Viewer

Look up the complete sentiment + classification profile for any restaurant.

In [24]:
def show_full_sentiment_profile(name: str) -> None:
    """
    Displays the complete sentiment and classification profile for one restaurant.
    Accepts partial, case-insensitive name matching.
    """
    matches = restaurants[restaurants['Name'].str.contains(name, case=False, na=False)]
    if matches.empty:
        print(f"⚠️  No restaurant found matching '{name}'.")
        return

    for _, row in matches.iterrows():
        print("=" * 62)
        print(f"  🍽️  {row['Name']}")
        print("=" * 62)
        print(f"  Location          : {row['Location']}")
        print(f"  Michelin          : {row['Michelin-Guide']}  |  Price: {row['Price']}  |  Rating: {row['Customer Ratings']}/5")
        print(f"  Description       : {row['Description']}")
        print()
        print(f"  ── Phase 3: Classifications ─────────────────────────")
        print(f"  Cuisine Group     : {row['simple_cuisine_group']}")
        print(f"  Dining Format     : {row['dining_format']}")
        print(f"  Predicted Occasion: {row['predicted_occasion']}  (conf: {row['occasion_confidence']:.3f})")
        print(f"  Predicted Vibe    : {row['predicted_vibe']}  (conf: {row['vibe_confidence']:.3f})")
        print()
        print(f"  ── Phase 4: Sentiment Analysis ──────────────────────")
        print(f"  Dominant Emotion  : {row['dominant_emotion']}")
        print(f"  Overall Sentiment : {row['overall_sentiment']}  (score: {row['sentiment_score']:.4f})")
        print(f"  Dining Mood       : {row['dining_mood']}")
        print()
        print(f"  ── Emotion Scores ───────────────────────────────────")
        emotion_data = [(label, row[f'emotion_{label}']) for label in EMOTION_LABELS]
        for label, score in sorted(emotion_data, key=lambda x: x[1], reverse=True):
            bar = '█' * int(score * 25)
            marker = " ◄ dominant" if label == row['dominant_emotion'] else ""
            print(f"    {label:<12} {bar:<8} {score:.4f}{marker}")
        print()


# Demo profiles across different restaurant types
show_full_sentiment_profile("Vespertine")    # Theatrical
show_full_sentiment_profile("71Above")       # Rooftop/views
show_full_sentiment_profile("Holbox")        # Casual Michelin
show_full_sentiment_profile("Hayato")        # Intimate omakase

  🍽️  Vespertine
  Location          : Culver City
  Michelin          : 2-Star  |  Price: $$$$  |  Rating: 5.0/5
  Description       : An avant-garde two-Michelin-star dining experience from Chef Jordan Kahn set inside a wavy obelisk architectural marvel with a sci-fi dreamscape atmosphere.

  ── Phase 3: Classifications ─────────────────────────
  Cuisine Group     : Contemporary American
  Dining Format     : Full Service
  Predicted Occasion: Special Occasion  (conf: 0.476)
  Predicted Vibe    : Hip & Trendy  (conf: 0.462)

  ── Phase 4: Sentiment Analysis ──────────────────────
  Dominant Emotion  : neutral
  Overall Sentiment : Positive  (score: 0.6789)
  Dining Mood       : Dramatic & Exciting

  ── Emotion Scores ───────────────────────────────────
    neutral      ███████████████████████ 0.9576 ◄ dominant
    joy          ███████████████ 0.6113
    surprise     ██       0.1166
    fear                  0.0286
    disgust               0.0264
    anger                 0.0259
  

## 🔁 Cell 23 — Update restaurant_metadata with Sentiment Labels

Appends the three new sentiment labels to `restaurant_metadata` so the  
ChromaDB vector database from Notebook 2 can also retrieve restaurants  
based on mood, emotion, and sentiment queries.

In [25]:
def enrich_metadata_with_sentiment(row: pd.Series) -> str:
    """
    Appends sentiment analysis labels to the restaurant_metadata string.
    Preserves all existing content; new labels are appended at the end.
    """
    base = str(row['restaurant_metadata']).rstrip('.')
    return (
        f"{base} "
        f"Dining Mood: {row['dining_mood']}. "
        f"Dominant Emotion: {row['dominant_emotion']}. "
        f"Overall Sentiment: {row['overall_sentiment']}."
    )

restaurants['restaurant_metadata'] = restaurants.apply(
    enrich_metadata_with_sentiment, axis=1
)

print("✅ restaurant_metadata enriched with sentiment labels.")
print("\nSample enriched metadata (Vespertine):")
vespertine_idx = restaurants[restaurants['Name'] == 'Vespertine'].index[0]
print(f"  {restaurants.loc[vespertine_idx, 'restaurant_metadata']}")
print("\nSample enriched metadata (Holbox):")
holbox_idx = restaurants[restaurants['Name'] == 'Holbox'].index[0]
print(f"  {restaurants.loc[holbox_idx, 'restaurant_metadata']}")

✅ restaurant_metadata enriched with sentiment labels.

Sample enriched metadata (Vespertine):
  Vespertine is a Contemporary American / Innovative restaurant located in Culver City, Los Angeles. An avant-garde two-Michelin-star dining experience from Chef Jordan Kahn set inside a wavy obelisk architectural marvel with a sci-fi dreamscape atmosphere. Price range: $$$$. Atmosphere: Fine-Dining. Michelin Guide: Michelin 2-Star. Customer Rating: 5.0/5 Cuisine Group: Contemporary American. Dining Format: Full Service. Best For: Special Occasion. Vibe: Hip & Trendy Dining Mood: Dramatic & Exciting. Dominant Emotion: neutral. Overall Sentiment: Positive.

Sample enriched metadata (Holbox):
  Holbox is a Mexican / Seafood restaurant located in South Los Angeles, Los Angeles. A Yucatecan-style seafood counter from Chef Gilberto Cetina Jr. inside Mercado La Paloma offering some of LA's most delicious and affordable Michelin-starred food. Price range: $$. Atmosphere: Casual. Michelin Guide: Miche

## 💾 Cell 24 — Export Final Enriched Dataset

In [ ]:
# Drop the temporary helper columns used only during inspection
helper_cols = ['desc_word_count', 'meta_char_count', 'meta_sent_count']
cols_to_drop = [c for c in helper_cols if c in restaurants.columns]
restaurants_final = restaurants.drop(columns=cols_to_drop)

OUTPUT_PATH = "../data/restaurants_with_emotions.csv"
restaurants_final.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Final dataset saved to: {OUTPUT_PATH}")
print(f"   Rows    : {len(restaurants_final)}")
print(f"   Columns : {len(restaurants_final.columns)}")
print(f"\nAll columns in final dataset:")

phase_map = {
    'restaurant_metadata':  'Phase 1+4 (enriched)',
    'simple_cuisine_group': 'Phase 3',
    'dining_format':        'Phase 3',
    'predicted_occasion':   'Phase 3',
    'occasion_confidence':  'Phase 3',
    'predicted_vibe':       'Phase 3',
    'vibe_confidence':      'Phase 3',
    'emotion_anger':        'Phase 4 ← NEW',
    'emotion_disgust':      'Phase 4 ← NEW',
    'emotion_fear':         'Phase 4 ← NEW',
    'emotion_joy':          'Phase 4 ← NEW',
    'emotion_neutral':      'Phase 4 ← NEW',
    'emotion_sadness':      'Phase 4 ← NEW',
    'emotion_surprise':     'Phase 4 ← NEW',
    'dominant_emotion':     'Phase 4 ← NEW',
    'overall_sentiment':    'Phase 4 ← NEW',
    'sentiment_score':      'Phase 4 ← NEW',
    'dining_mood':          'Phase 4 ← NEW',
}

for i, col in enumerate(restaurants_final.columns, 1):
    tag = phase_map.get(col, 'Phase 1–2 (original)')
    print(f"   {i:>2}. {col:<35} [{tag}]")

✅ Final dataset saved to: restaurants_with_emotions.csv
   Rows    : 71
   Columns : 32

All columns in final dataset:
    1. Name                                [Phase 1–2 (original)]
    2. Location                            [Phase 1–2 (original)]
    3. Description                         [Phase 1–2 (original)]
    4. Address                             [Phase 1–2 (original)]
    5. Telephone Number                    [Phase 1–2 (original)]
    6. Price                               [Phase 1–2 (original)]
    7. Cuisine Type                        [Phase 1–2 (original)]
    8. Dining Atmosphere                   [Phase 1–2 (original)]
    9. Sky-High Rooftop                    [Phase 1–2 (original)]
   10. Michelin-Guide                      [Phase 1–2 (original)]
   11. Customer Ratings                    [Phase 1–2 (original)]
   12. Operation Hours                     [Phase 1–2 (original)]
   13. Reservations                        [Phase 1–2 (original)]
   14. Dress Code      

## 📊 Cell 25 — Final Pipeline Summary Report

In [27]:
print("=" * 66)
print("   SENTIMENT ANALYSIS PIPELINE — FINAL SUMMARY")
print("=" * 66)
print(f"  Total Restaurants          : {len(restaurants_final)}")
print(f"  New Columns Added (Phase 4): 10")
print(f"  Total Columns in Output    : {len(restaurants_final.columns)}")
print()

print("  Layer 1 — Emotion Scores (j-hartmann model):")
for label, count in restaurants_final['dominant_emotion'].value_counts().items():
    avg = restaurants_final[f'emotion_{label}'].mean()
    print(f"    dominant_{label:<12}: {count:>3} restaurants  (avg score: {avg:.4f})")
print()

print("  Layer 2 — Sentiment Tone (cardiffnlp model):")
for label, count in restaurants_final['overall_sentiment'].value_counts().items():
    avg_score = restaurants_final[restaurants_final['overall_sentiment'] == label]['sentiment_score'].mean()
    avg_rating = restaurants_final[restaurants_final['overall_sentiment'] == label]['Customer Ratings'].mean()
    print(f"    {label:<12}: {count:>3} restaurants  (avg conf: {avg_score:.4f}, avg rating: {avg_rating:.2f})")
print()

print("  Layer 3 — Dining Mood (keyword-based):")
for mood, count in restaurants_final['dining_mood'].value_counts().items():
    print(f"    {mood:<28}: {count:>3} restaurants")
print()

print(f"  Output: restaurants_with_emotions.csv")
print("=" * 66)
print("  ✅ Full sentiment enrichment complete!")
print("=" * 66)
print()
print("🚀 NEXT STEP: Notebook 5 — Gradio Dashboard UI")
print("   Input: restaurants_with_emotions.csv")
print("   New UI filters to build from Phase 4 columns:")
print("     • Dining Mood    (dining_mood)")
print("     • Emotion Filter (dominant_emotion)")
print("     • Sentiment      (overall_sentiment)")
print("   Combine with Phase 3 filters:")
print("     • Cuisine Group  (simple_cuisine_group)")
print("     • Dining Format  (dining_format)")
print("     • Occasion       (predicted_occasion)")
print("     • Vibe           (predicted_vibe)")

   SENTIMENT ANALYSIS PIPELINE — FINAL SUMMARY
  Total Restaurants          : 71
  New Columns Added (Phase 4): 10
  Total Columns in Output    : 32

  Layer 1 — Emotion Scores (j-hartmann model):
    dominant_neutral     :  65 restaurants  (avg score: 0.9323)
    dominant_joy         :   6 restaurants  (avg score: 0.7344)

  Layer 2 — Sentiment Tone (cardiffnlp model):
    Positive    :  50 restaurants  (avg conf: 0.7560, avg rating: 4.57)
    Neutral     :  21 restaurants  (avg conf: 0.6570, avg rating: 4.41)

  Layer 3 — Dining Mood (keyword-based):
    Elegant & Refined           :  23 restaurants
    Warm & Inviting             :  18 restaurants
    Dramatic & Exciting         :  15 restaurants
    Intimate & Personal         :  13 restaurants
    Lively & Energetic          :   2 restaurants

  Output: restaurants_with_emotions.csv
  ✅ Full sentiment enrichment complete!

🚀 NEXT STEP: Notebook 5 — Gradio Dashboard UI
   Input: restaurants_with_emotions.csv
   New UI filters to bu